# FinAdvisor — Session 1 Demo

**Theme:** *We can talk to an open-weights LLM and pull live finance data. Nothing is grounded yet — that's the point.*

This notebook is the live demo surface for Mentored Session 1. It exercises the two lowest layers of the stack:

1. **Alpha Vantage wrapper** — cached HTTP calls that return live market data.
2. **LLM client** — a thin wrapper around Hugging Face Inference Providers (Llama-3.3-70B via Groq).

There is intentionally **no RAG, no tools, no agent loop** in this notebook. Those come in Sessions 2 and 3. The gap this notebook exposes — *the LLM sounds confident but nothing is grounded* — is the motivation for Session 2.

**Pre-flight:** `.env` populated with `HF_TOKEN` and `ALPHA_VANTAGE_KEY`; run from the repo root or with the src path on `sys.path`.

In [ ]:
# Make `advisor.*` importable when running from notebooks/ without an editable install.
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from advisor.config import settings
print('Model:   ', settings.llm_model_id)
print('Provider:', settings.llm_provider)
print('Chroma:  ', settings.chroma_dir)

## 1. Live market data via Alpha Vantage

`get_quote` calls `GLOBAL_QUOTE` under the hood. Every response is cached to `data/raw/av_cache.sqlite` with a 30-minute TTL, so repeated runs in this notebook do not burn the 25-req/day free-tier quota.

In [ ]:
from advisor.tools.alpha_vantage import get_quote, get_company_overview

quote = get_quote('AAPL')
quote

In [ ]:
# Second call should be a cache hit — near-instant, no HTTP round-trip.
import time
t0 = time.perf_counter()
_ = get_quote('AAPL')
print(f'Second call: {(time.perf_counter() - t0) * 1000:.1f} ms (cache hit)')

## 2. Talk to the LLM

This is our *baseline*: hand the LLM the raw quote JSON and let it write a paragraph. No RAG, no tools — just a system prompt, a user message, and a chat completion.

In [ ]:
from advisor.llm.client import chat
from advisor.llm.prompts import SYSTEM_PROMPT, format_persona

system = SYSTEM_PROMPT.format(persona_block=format_persona({}))
print(system[:600] + '\n\n...[truncated for display]')

In [ ]:
user_msg = (
    f'Live quote for AAPL: {quote}. '
    'Summarise Apple\'s current position for a moderately risk-tolerant investor. '
    'Keep it to 5 sentences.'
)

resp = chat(
    messages=[
        {'role': 'system', 'content': system},
        {'role': 'user',   'content': user_msg},
    ],
    temperature=0.2,
    max_tokens=400,
)
print(resp.choices[0].message.content)

## 3. Richer prompt — hand the model the company overview too

`get_company_overview` returns sector, P/E, market cap, beta, dividends, etc. With richer context the LLM's answer is more specific — but every claim is still ungrounded from the *user's* perspective. They cannot verify where any number came from.

In [ ]:
overview = get_company_overview('AAPL')
snapshot = {k: overview.get(k) for k in ('Symbol', 'Sector', 'PERatio', 'MarketCapitalization', 'Beta', 'DividendYield')}
snapshot

In [ ]:
user_msg = (
    f'Live quote: {quote}\n'
    f'Company snapshot: {snapshot}\n\n'
    'Given the profile of a moderately risk-tolerant, retirement-focused 30-year-old, '
    'discuss whether Apple fits their portfolio. Do not give a directive; frame as considerations.'
)

resp = chat(
    messages=[
        {'role': 'system', 'content': system},
        {'role': 'user',   'content': user_msg},
    ],
    temperature=0.2,
    max_tokens=500,
)
print(resp.choices[0].message.content)

## 4. The gap — where did those numbers come from?

Read the paragraph above carefully. The model probably cited a P/E, a sector, maybe a beta — and it may be right. But:

- **Nothing tells the user which claims came from `snapshot` vs. from the model's parametric memory.**
- **Nothing prevents the model from inventing a number if the snapshot were missing a field.**
- **The model has no access to definitions** — e.g. *what is a Roth IRA?*, *what is a good P/E for a mega-cap tech stock?* — beyond what its pretraining happens to remember.

This is the failure mode a finance assistant cannot afford. Sessions 2 fixes it with retrieval-augmented grounding: every claim gets a `[Source: …]` marker or falls back to *I don't have data on this*.

**Next session:** the same MSFT / Apple prompt, but every quantitative claim is auditable back to a source document.

---

### Session 1 talking points cheat-sheet

- Open-weights via HF Inference Providers keeps the FinGPT thesis intact; provider switch is a `.env` line.
- Alpha Vantage covers the whole surface (quotes, overview, news sentiment, technicals, sectors, FX) with a single credential; caching neutralises the 25/day free-tier ceiling.
- We deferred fine-tuning; inference-side domain adaptation via prompt + RAG + tools composes cleanly with LoRA later.
- Single `Settings` object reads `.env` — one source of truth, credentials never in code or logs.